# FLOWER FEDERATED LEARNING SETUP
# Client Implementation

In [6]:
# DIAGNOSTIC CELL — run alone, share the full output
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print()

# Check where pip is installing to
!pip show flwr 2>/dev/null || echo "flwr NOT installed"
!which pip
!which python
print()

# Check sys.path
import sys
for p in sys.path:
    print("PATH:", p)

!{sys.executable} -c "import flwr; print('✅ flwr version:', flwr.__version__)"

Python executable: /usr/bin/python3
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

flwr NOT installed
/usr/local/bin/pip
/usr/local/bin/python

PATH: /content
PATH: /env/python
PATH: /usr/lib/python312.zip
PATH: /usr/lib/python3.12
PATH: /usr/lib/python3.12/lib-dynload
PATH: 
PATH: /usr/local/lib/python3.12/dist-packages
PATH: /usr/lib/python3/dist-packages
PATH: /usr/local/lib/python3.12/dist-packages/IPython/extensions
PATH: /root/.ipython
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'flwr'


In [1]:
# ============================================
# CELL 1: Setup Environment (NO FLOWER NEEDED)
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/FL-Pneumonia-Detection'
os.chdir(PROJECT_ROOT)

import sys
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.models as models
import wandb
import json
from datetime import datetime
from collections import OrderedDict

print(f"✅ NumPy version: {np.__version__}")
print(f"✅ PyTorch version: {torch.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
print(f"✅ Environment ready — using manual FedAvg (no Flower dependency)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ NumPy version: 2.4.3
✅ PyTorch version: 2.10.0+cu128
✅ Device: cuda
✅ Environment ready — using manual FedAvg (no Flower dependency)


In [2]:
# ============================================
# CELL 2: Initialize WandB
# ============================================
wandb.login()

wandb.init(
    project="fl-pneumonia-detection",
    name="federated-simulation-fedavg",
    config={
        "phase": "federated_learning",
        "aggregation": "FedAvg (manual)",
        "num_rounds": 10,
        "num_clients": 3,
        "local_epochs": 5,
        "learning_rate": 0.001
    }
)

print("✅ WandB initialized!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chiwa-vw (chiwa-vw-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ WandB initialized!


In [3]:
# ============================================
# Initialize WandB
# ============================================
wandb.login()

wandb.init(
    project="fl-pneumonia-detection",
    name="week2-day6-flower-client-setup",
    config={
        "phase": "federated_learning_setup",
        "week": 2,
        "day": 6,
        "framework": "Flower 1.11.1",
        "num_hospitals": 3,
        "strategy": "FedAvg"
    }
)

print("✅ WandB initialized!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chiwa-vw (chiwa-vw-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:279: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:279: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


✅ WandB initialized!


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:174: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


In [3]:
# ============================================
# CELL 3: Load Utilities & Model
# ============================================
sys.path.append(PROJECT_ROOT)

from utils.preprocessing import (
    ChestXrayDataset,
    get_train_transforms,
    get_val_test_transforms
)

def create_model(num_classes=2):
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.DEFAULT
    )
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, num_classes)
    )
    return model

print("✅ Model function defined")

✅ Model function defined


In [4]:
# ============================================
# CELL 4: PneumoniaClient (unchanged from Day 6)
# ============================================
class PneumoniaClient:
    """
    Hospital client for federated learning.
    Implements FedAvg client logic without Flower dependency.
    """

    def __init__(self, hospital_id, data_dir, model, device, epochs=5):
        self.hospital_id = hospital_id
        self.device = device
        self.epochs = epochs

        self.train_dataset = ChestXrayDataset(
            data_dir=data_dir,
            split='',
            transform=get_train_transforms()
        )

        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size=32,
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )

        self.model = model.to(device)
        self.criterion = nn.CrossEntropyLoss()

        print(f"✅ Hospital {hospital_id}: {len(self.train_dataset)} samples")

    def get_parameters(self):
        return [val.cpu().numpy()
                for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict(
            {k: torch.tensor(v) for k, v in params_dict}
        )
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        epochs = config.get("local_epochs", self.epochs)
        lr = config.get("learning_rate", 0.001)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

        self.model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for epoch in range(epochs):
            for images, labels in self.train_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_loss = total_loss / (len(self.train_dataset) * epochs)
        avg_acc  = correct / total

        return self.get_parameters(), len(self.train_dataset), {
            "loss": avg_loss,
            "accuracy": avg_acc
        }

    def evaluate(self, parameters):
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.train_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                total_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return total_loss / len(self.train_dataset), {
            "accuracy": correct / total
        }

print("✅ PneumoniaClient defined!")

✅ PneumoniaClient defined!


In [5]:
# ============================================
# CELL 5: Manual FedAvg Simulation
# ============================================
def federated_avg(client_params_list, client_sizes):
    """Weighted average of client parameters (FedAvg)"""
    total = sum(client_sizes)
    avg_params = []
    for layers in zip(*client_params_list):
        weighted = sum(
            p * (n / total)
            for p, n in zip(layers, client_sizes)
        )
        avg_params.append(weighted)
    return avg_params


def run_federated_simulation(num_rounds=10, num_clients=3, strategy='iid'):
    """
    Full FedAvg simulation across simulated hospitals.
    Mathematically identical to Flower's FedAvg implementation.
    """
    DATA_SPLITS = os.path.join(PROJECT_ROOT, 'data/splits')

    # Initialize global model
    global_model  = create_model(num_classes=2)
    global_params = [v.cpu().numpy()
                     for v in global_model.state_dict().values()]

    history = []

    for round_num in range(1, num_rounds + 1):
        print(f"\n{'='*55}")
        print(f"  ROUND {round_num}/{num_rounds}")
        print(f"{'='*55}")

        client_params_list = []
        client_sizes       = []
        round_metrics      = []

        for cid in range(num_clients):
            print(f"\n🏥 Hospital {cid} training...")
            data_dir = os.path.join(
                DATA_SPLITS, strategy, f'hospital_{cid}'
            )
            client = PneumoniaClient(
                hospital_id=cid,
                data_dir=data_dir,
                model=create_model(num_classes=2),
                device=device,
                epochs=5
            )

            updated_params, n, metrics = client.fit(
                global_params,
                {"local_epochs": 5,
                 "learning_rate": 0.001 * (0.95 ** round_num)}
            )

            client_params_list.append(updated_params)
            client_sizes.append(n)
            round_metrics.append(metrics)

            print(f"   Loss: {metrics['loss']:.4f} | "
                  f"Acc:  {metrics['accuracy']:.4f} | "
                  f"Samples: {n}")

        # Aggregate
        global_params = federated_avg(client_params_list, client_sizes)

        avg_loss = np.mean([m['loss']     for m in round_metrics])
        avg_acc  = np.mean([m['accuracy'] for m in round_metrics])

        print(f"\n📊 Round {round_num} Global Average:")
        print(f"   Loss:     {avg_loss:.4f}")
        print(f"   Accuracy: {avg_acc:.4f}")

        history.append({
            'round':        round_num,
            'avg_loss':     float(avg_loss),
            'avg_accuracy': float(avg_acc)
        })

        wandb.log({
            'round':        round_num,
            'avg_loss':     avg_loss,
            'avg_accuracy': avg_acc
        })

    return global_params, history


# Run it
final_params, history = run_federated_simulation(
    num_rounds=10,
    num_clients=3,
    strategy='iid'
)

print("\n✅ Federated simulation complete!")
print(f"Final round accuracy: {history[-1]['avg_accuracy']:.4f}")

# Save history
history_path = os.path.join(PROJECT_ROOT, 'results/metrics/fl_history.json')
os.makedirs(os.path.dirname(history_path), exist_ok=True)
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"✅ History saved to {history_path}")

wandb.finish()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 161MB/s]



  ROUND 1/10

🏥 Hospital 0 training...
✅ Hospital 0: 1738 samples
   Loss: 0.1091 | Acc:  0.9591 | Samples: 1738

🏥 Hospital 1 training...
✅ Hospital 1: 1738 samples
   Loss: 0.1075 | Acc:  0.9595 | Samples: 1738

🏥 Hospital 2 training...
✅ Hospital 2: 1740 samples
   Loss: 0.1143 | Acc:  0.9557 | Samples: 1740

📊 Round 1 Global Average:
   Loss:     0.1103
   Accuracy: 0.9581

  ROUND 2/10

🏥 Hospital 0 training...
✅ Hospital 0: 1738 samples
   Loss: 0.0666 | Acc:  0.9765 | Samples: 1738

🏥 Hospital 1 training...
✅ Hospital 1: 1738 samples
   Loss: 0.0673 | Acc:  0.9766 | Samples: 1738

🏥 Hospital 2 training...
✅ Hospital 2: 1740 samples
   Loss: 0.0673 | Acc:  0.9756 | Samples: 1740

📊 Round 2 Global Average:
   Loss:     0.0670
   Accuracy: 0.9763

  ROUND 3/10

🏥 Hospital 0 training...
✅ Hospital 0: 1738 samples
   Loss: 0.0460 | Acc:  0.9831 | Samples: 1738

🏥 Hospital 1 training...
✅ Hospital 1: 1738 samples
   Loss: 0.0415 | Acc:  0.9849 | Samples: 1738

🏥 Hospital 2 training..

avg_accuracy,▁▄▆▆▇▇████
avg_loss,█▅▄▃▂▂▁▁▁▁
round,▁▂▃▃▄▅▆▆▇█
avg_accuracy,0.99632
avg_loss,0.01009
round,10


In [6]:

# ============================================
#Test Client Methods
# ============================================
print("\n🧪 Testing client methods...")

# Test get_parameters
print("\n1. Testing get_parameters()...")
params = client_0.get_parameters(config={})
print(f"   ✅ Retrieved {len(params)} parameter arrays")
print(f"   First param shape: {params[0].shape}")

# Test set_parameters
print("\n2. Testing set_parameters()...")
client_0.set_parameters(params)
print(f"   ✅ Parameters set successfully")

# Test fit (local training)
print("\n3. Testing fit() - Local training...")
print("   This will take ~2-3 minutes for 2 epochs...")

config = {
    "local_epochs": 2,
    "learning_rate": 0.001
}

updated_params, num_examples, metrics = client_0.fit(params, config)

print(f"\n   ✅ Training complete!")
print(f"   Samples trained on: {num_examples}")
print(f"   Final loss: {metrics['loss']:.4f}")
print(f"   Final accuracy: {metrics['accuracy']:.4f}")
print(f"   Updated parameters: {len(updated_params)} arrays")

# Test evaluate
print("\n4. Testing evaluate()...")
loss, num_examples, eval_metrics = client_0.evaluate(updated_params, config={})
print(f"   ✅ Evaluation complete!")
print(f"   Loss: {loss:.4f}")
print(f"   Accuracy: {eval_metrics['accuracy']:.4f}")


🧪 Testing client methods...

1. Testing get_parameters()...


NameError: name 'client_0' is not defined

In [ ]:

# ============================================
#Create Client Factory Function
# ============================================
def create_client_fn(hospital_id, strategy='iid', epochs=5):
    """
    Factory function to create a client

    Args:
        hospital_id: Hospital identifier (0, 1, 2)
        strategy: 'iid' or 'non_iid'
        epochs: Local training epochs per round

    Returns:
        Flower client instance
    """
    # Data path
    data_dir = os.path.join(DATA_SPLITS, strategy, f'hospital_{hospital_id}')

    # Create fresh model
    model = create_model(num_classes=2)

    # Create and return client
    return PneumoniaClient(
        hospital_id=hospital_id,
        data_dir=data_dir,
        model=model,
        device=device,
        epochs=epochs
    )

print("\n✅ Client factory function defined!")

# Test factory
print("\n🧪 Testing client factory...")
test_client = create_client_fn(hospital_id=1, strategy='iid', epochs=2)
print(f"   ✅ Created client for Hospital 1")
print(f"   Training samples: {len(test_client.train_dataset)}")

In [ ]:

# ============================================
#Save Client Configuration
# ============================================
client_config = {
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "week": 2,
    "day": 6,
    "milestone": "Flower Client Implementation Complete",
    "framework": {
        "name": "Flower",
        "version": "1.11.1",
        "client_type": "NumPyClient"
    },
    "client_configuration": {
        "local_epochs": 5,
        "batch_size": 32,
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "model": "EfficientNetB0"
    },
    "hospitals": {
        "total": 3,
        "hospital_0_samples": len(client_0.train_dataset) if 'client_0' in locals() else "Unknown",
        "hospital_1_samples": len(test_client.train_dataset) if 'test_client' in locals() else "Unknown"
    },
    "next_steps": [
        "Day 7: Implement Flower Server",
        "Day 8: Test federated training with IID data",
        "Day 9: Compare centralized vs federated performance"
    ]
}

# Save config
config_path = os.path.join(PROJECT_ROOT, 'federated', 'client_config.json')
os.makedirs(os.path.join(PROJECT_ROOT, 'federated'), exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(client_config, f, indent=2)

print(f"\n✅ Client configuration saved to: {config_path}")


In [ ]:

# ============================================
#Save Client Code as Module
# ============================================
client_code = '''"""
Flower Client for Federated Pneumonia Detection
Week 2, Day 6
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.models as models
import flwr as fl
from collections import OrderedDict
import sys
import os

# Add project root to path
PROJECT_ROOT = '/content/drive/MyDrive/FL-Pneumonia-Detection'
sys.path.append(PROJECT_ROOT)

from utils.preprocessing import ChestXrayDataset, get_train_transforms

def create_model(num_classes=2):
    """Create EfficientNetB0 model"""
    model = models.efficientnet_b0(pretrained=True)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, num_classes)
    )
    return model

class PneumoniaClient(fl.client.NumPyClient):
    """Flower client for federated learning"""

    def __init__(self, hospital_id, data_dir, model, device, epochs=5):
        self.hospital_id = hospital_id
        self.device = device
        self.epochs = epochs

        self.train_dataset = ChestXrayDataset(
            data_dir=data_dir,
            split='',
            transform=get_train_transforms()
        )

        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size=32,
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )

        self.model = model.to(device)
        self.criterion = nn.CrossEntropyLoss()

        print(f"✅ Hospital {hospital_id}: {len(self.train_dataset)} samples")

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        epochs = config.get("local_epochs", self.epochs)
        lr = config.get("learning_rate", 0.001)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

        self.model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for epoch in range(epochs):
            for images, labels in self.train_loader:
                images, labels = images.to(self.device), labels.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_loss = total_loss / (len(self.train_dataset) * epochs)
        avg_acc = correct / total

        return (
            self.get_parameters(config={}),
            len(self.train_dataset),
            {"loss": avg_loss, "accuracy": avg_acc}
        )

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()

        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.train_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                total_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_loss = total_loss / len(self.train_dataset)
        accuracy = correct / total

        return avg_loss, len(self.train_dataset), {"accuracy": accuracy}
'''

# Save client code
client_code_path = os.path.join(PROJECT_ROOT, 'federated', 'client.py')
with open(client_code_path, 'w') as f:
    f.write(client_code)

print(f"✅ Client code saved to: {client_code_path}")



In [1]:

# ============================================
#Summary
# ============================================
summary_table = wandb.Table(
    columns=["Component", "Status"],
    data=[
        ["Flower Framework", "✅ Installed (v1.11.1)"],
        ["Client Class", "✅ Implemented"],
        ["get_parameters()", "✅ Tested"],
        ["set_parameters()", "✅ Tested"],
        ["fit() method", "✅ Tested (local training)"],
        ["evaluate() method", "✅ Tested"],
        ["Client Factory", "✅ Created"],
        ["Code Module", "✅ Saved to federated/client.py"]
    ]
)

wandb.log({"day6_summary": summary_table})

print("\n" + "="*60)
print("🎉 WEEK 2, DAY 6 COMPLETE!")
print("="*60)
print("\n✅ Accomplishments:")
print("   • Flower framework installed and configured")
print("   • PneumoniaClient class implemented")
print("   • All client methods tested successfully")
print("   • Client factory function created")
print("   • Code saved as reusable module")
print(f"\n📁 Files created:")
print(f"   • federated/client.py")
print(f"   • federated/client_config.json")
print(f"\n✅ Next: Day 7 - Implement Flower Server")
print("="*60)

wandb.finish()

NameError: name 'wandb' is not defined